## Step 1 · Install Dependencies

In [ ]:
!pip install -q groq crewai langchain_groq
print('✅ Dependencies ready')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.3/142.3 kB 3.7 MB/s eta 0:00:00
✅ Dependencies ready


## Step 2 · Configuration
> Add your Groq AI API key to Colab Secrets (🔑 icon) with name `GROQ_API_KEY`

In [ ]:
import os, json, re, time
import pandas as pd
from groq import Groq
from datetime import datetime
from typing import Dict, List
from google.colab import userdata

# ── API Key ───────────────────────────────────────────────────────────────────
GROQ_API_KEY = userdata.get('GROQ_API_KEY')
if not GROQ_API_KEY:
    raise ValueError('GROQ_API_KEY not found in Colab Secrets.')

# ── Settings ──────────────────────────────────────────────────────────────────
MODEL_NAME = 'llama-3.1-8b-instant'

PRIORITY_RULES = {
    'Bug':             {'default': 'High',
                        'keywords_critical': ['crash','data loss','startup','unusable']},
    'Feature Request': {'default': 'Medium'},
    'Complaint':       {'default': 'Medium'},
    'Praise':          {'default': 'Low'},
    'Spam':            {'default': 'Low'},
}

TICKETS_OUTPUT = 'generated_tickets.csv'
LOG_OUTPUT     = 'processing_log.csv'
METRICS_OUTPUT = 'metrics.csv'

client = Groq(api_key=GROQ_API_KEY)
print(f'✅ Groq configured: {MODEL_NAME}')

✅ Groq configured: llama-3.1-8b-instant


## Step 3 · Agent Definitions

In [ ]:
import os
import json
import pandas as pd
from crewai import Agent, Task, Crew, Process
from langchain_groq import ChatGroq

# 1. LLM Setup
llm = ChatGroq(
    temperature=0,
    model_name="llama-3.1-8b-instant",
    groq_api_key=GROQ_API_KEY
)

# 2. CrewAI Agents
classifier_agent = Agent(
    role='Feedback Classifier',
    goal='Categorize feedback into Bug, Feature Request, Praise, Complaint, or Spam',
    backstory='You are an expert product analyst categorizing user feedback.',
    allow_delegation=False,
    llm=llm
)

bug_analyst_agent = Agent(
    role='Bug Analysis Agent',
    goal='Extract technical details like steps to reproduce, platform info, and severity from bugs.',
    backstory='You are a software QA engineer focusing on system stability.',
    allow_delegation=False,
    llm=llm
)

ticket_creator_agent = Agent(
    role='Ticket Creator Agent',
    goal='Generate structured engineering tickets with priority and effort estimates.',
    backstory='You are a technical lead creating actionable tasks for developers.',
    allow_delegation=True,
    llm=llm
)

quality_critic_agent = Agent(
    role='Quality Critic Agent',
    goal='Review generated tickets for completeness and output as strict JSON.',
    backstory='You are a senior auditor ensuring every ticket meets quality standards.',
    allow_delegation=False,
    llm=llm
)

# 3. Tasks setup for testing inside the notebook
classify_task = Task(
    description="Analyze user feedback and categorize it.",
    expected_output="Categorized feedback.",
    agent=classifier_agent
)

extract_tech_task = Task(
    description="Extract technical details for Bugs and Feature Requests.",
    expected_output="Technical details and impact estimations.",
    agent=bug_analyst_agent
)

ticket_creation_task = Task(
    description="Create structured engineering tickets.",
    expected_output="A list of engineering tickets.",
    agent=ticket_creator_agent
)

quality_review_task = Task(
    description="Review tickets and output ONLY a valid JSON array. Schema: [{\"ticket_id\": \"TKT-123\", \"category\": \"Bug\", \"priority\": \"High\", \"title\": \"...\", \"quality_score\": 8, \"qa_approved\": true}]",
    expected_output="A strict JSON array containing final tickets.",
    agent=quality_critic_agent
)

# 4. Crew Assembly
feedback_crew = Crew(
    agents=[classifier_agent, bug_analyst_agent, ticket_creator_agent, quality_critic_agent],
    tasks=[classify_task, extract_tech_task, ticket_creation_task, quality_review_task],
    process=Process.sequential,
    verbose=True
)

print('✅ CrewAI Engine Ready!')

## Step 4 · Upload CSV Files

In [ ]:
from google.colab import files
print('📂 Upload: app_store_reviews.csv, support_emails.csv, expected_classifications.csv')
uploaded = files.upload()
print(f'\n✅ Uploaded: {list(uploaded.keys())}')

📂 Upload: app_store_reviews.csv, support_emails.csv, expected_classifications.csv


Saving app_store_reviews.csv to app_store_reviews.csv
Saving expected_classifications.csv to expected_classifications.csv
Saving support_emails.csv to support_emails.csv

✅ Uploaded: ['app_store_reviews.csv', 'expected_classifications.csv', 'support_emails.csv']


## Step 5 · Run the Pipeline

In [ ]:
import pandas as pd
import json
import time

print("📖 Reading uploaded CSVs...")
try:
    df_r = pd.read_csv('app_store_reviews.csv')
    df_e = pd.read_csv('support_emails.csv')
    
    # Data ko text format mein convert karna CrewAI ke liye
    feedback_text_block = ""
    for _, row in df_r.iterrows():
        feedback_text_block += f"ID: {row.get('review_id', 'N/A')} | Type: App Store | Text: {str(row.get('review_text', ''))[:300]}\n"
    for _, row in df_e.iterrows():
        feedback_text_block += f"ID: {row.get('email_id', 'N/A')} | Type: Support Email | Text: {str(row.get('body', ''))[:300]}\n"
        
    print(f"✅ Loaded {len(df_r)} reviews and {len(df_e)} emails.")
    
    # Task 1 ke description mein dynamically data inject karna
    classify_task.description = f"Analyze the following user feedback items carefully:\n\n{feedback_text_block}\n\nCategorize each item strictly into Bug, Feature Request, Praise, Complaint, or Spam."
    
    print("🚀 CrewAI processing started. Please wait...")
    start_time = time.time()
    
    # CrewAI execute karna
    result = feedback_crew.kickoff()
    
    # JSON Parse karna aur CSV mein save karna (taaki Step 6 aur 7 chal sakein)
    cleaned_result = result.raw.strip().strip('```json').strip('
```')
    tickets = json.loads(cleaned_result)
    
    df_tickets = pd.DataFrame(tickets)
    df_tickets.to_csv(TICKETS_OUTPUT, index=False)
    
    print(f"\n✅ Pipeline Complete in {round(time.time() - start_time, 1)}s")
    print(f"💾 Saved {TICKETS_OUTPUT} ({len(tickets)} tickets)")
    
except Exception as e:
    print(f"❌ Error during execution: {e}")


📖 [CSVReaderAgent] Reading files...
  ✅ 5 reviews + 3 emails = 8 items

🏷️  [ClassifierAgent] Classifying 8 items in 1 API call...
    R001   → Bug (0.90)
    R002   → Bug (0.90)
    R003   → Praise (0.80)
    R004   → Feature Request (0.90)
    R005   → Spam (1.00)
    E001   → Bug (0.90)
    E002   → Bug (0.90)
    E003   → Feature Request (0.80)
  ✅ Classified 8/8

🔍 [BugAnalysisAgent] Extracting technical details from Bug items...
    R001   → severity=Critical platform=Google Play
    R002   → severity=High platform=App Store
    E001   → severity=Critical platform=Android
    E002   → severity=High platform=Ios
  ✅ Analyzed 4 bug(s)

💡 [FeatureExtractorAgent] Estimating impact for Feature Request items...
    R004   → demand=High impact=Medium
    E003   → demand=High impact=Medium
  ✅ Extracted 2 feature request(s)

🎫 [TicketCreatorAgent] Creating 8 tickets in 1 API call...
  ✅ TKT-R001     Bug              Critical Q=9
  ✅ TKT-R002     Bug              High     Q=8
  ✅ TKT-R00

## Step 6 · Accuracy Evaluation

In [ ]:
from IPython.display import display

expected_df  = pd.read_csv('expected_classifications.csv')
generated_df = pd.read_csv(TICKETS_OUTPUT)

merged = generated_df.merge(
    expected_df[['source_id','category','priority']].rename(
        columns={'category':'expected_category','priority':'expected_priority'}),
    on='source_id', how='inner')

total = len(merged)
if total == 0:
    print('⚠️  No source_id matches found.')
else:
    cat_ok = (merged['category'] == merged['expected_category']).sum()
    pri_ok = (merged['priority']  == merged['expected_priority']).sum()
    print(f'\n📊 Accuracy Report ({total} items)')
    print(f'  Category : {cat_ok}/{total} = {100*cat_ok/total:.1f}%')
    print(f'  Priority : {pri_ok}/{total} = {100*pri_ok/total:.1f}%')
    miss = merged[merged['category'] != merged['expected_category']]
    if miss.empty:
        print('\n🎉 Perfect category match!')
    else:
        print(f'\n⚠️  {len(miss)} mismatches:')
        for _, r in miss.iterrows():
            print(f'    {r["source_id"]}: got={r["category"]}  expected={r["expected_category"]}')
    display(merged[['source_id','category','expected_category','priority','expected_priority']])


📊 Accuracy Report (8 items)
  Category : 8/8 = 100.0%
  Priority : 8/8 = 100.0%

🎉 Perfect category match!


,source_id,category,expected_category,priority,expected_priority
0,E001,Bug,Bug,Critical,Critical
1,R001,Bug,Bug,Critical,Critical
2,E002,Bug,Bug,High,High
3,R002,Bug,Bug,High,High
4,E003,Feature Request,Feature Request,Medium,Medium
5,R004,Feature Request,Feature Request,Medium,Medium
6,R003,Praise,Praise,Low,Low
7,R005,Spam,Spam,Low,Low


## Step 7 · Inspect Tickets

In [ ]:
from IPython.display import display

df = pd.read_csv(TICKETS_OUTPUT)
print(f'Total tickets: {len(df)}')
print('\nCategory counts:')
print(df['category'].value_counts().to_string())
print('\nPriority counts:')
print(df['priority'].value_counts().to_string())
print(f'\nAvg quality: {df["quality_score"].mean():.2f}/10')
display(df[['ticket_id','category','priority','title','quality_score','qa_approved']])

Total tickets: 8

Category counts:
category
Bug                4
Feature Request    2
Praise             1
Spam               1

Priority counts:
priority
Critical    2
High        2
Medium      2
Low         2

Avg quality: 5.50/10


,ticket_id,category,priority,title,quality_score,qa_approved
0,TKT-E001,Bug,Critical,[BUG] App Crashes on Launch on Samsung Galaxy S22,9,True
1,TKT-R001,Bug,Critical,[BUG] App Crashes on Tapping My Tasks,9,True
2,TKT-E002,Bug,High,[BUG] Login Fails with Blank White Screen on i...,8,False
3,TKT-R002,Bug,High,[BUG] Login Fails with Blank White Screen,8,False
4,TKT-E003,Feature Request,Medium,[FEATURE] Add Dark Mode Option Based on User F...,6,False
5,TKT-R004,Feature Request,Medium,[FEATURE] Add Dark Mode Option,2,False
6,TKT-R003,Praise,Low,[PRAISE] App is Amazing and User-Friendly,1,True
7,TKT-R005,Spam,Low,[SPAM] Unauthorized Advertising,1,False


## Step 8 · Download Outputs

In [ ]:
from google.colab import files
for fp in [TICKETS_OUTPUT, LOG_OUTPUT, METRICS_OUTPUT]:
    if os.path.exists(fp):
        files.download(fp)
        print(f'⬇️  {fp}')
    else:
        print(f'⚠️  {fp} not found — run the pipeline first (Step 6).')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

⬇️  generated_tickets.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

⬇️  processing_log.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

⬇️  metrics.csv


## Step 9 · Launch Streamlit UI
Run after the pipeline completes. Requires a free ngrok token from [dashboard.ngrok.com](https://dashboard.ngrok.com)  
Add it as Colab Secret `NGROK_TOKEN`.

In [ ]:
!pip install -q streamlit pyngrok
import os, threading, time, urllib.request, urllib.error
from google.colab import userdata
from pyngrok import ngrok, conf

# ── Get ngrok token ───────────────────────────────────────────────────────────
NGROK_TOKEN = ''
try:
    NGROK_TOKEN = userdata.get('NGROK_TOKEN') or ''
except Exception:
    pass
if not NGROK_TOKEN:
    NGROK_TOKEN = input('Paste your ngrok authtoken: ').strip()

# ── Upload streamlit_app.py AND crew_agents.py if not present ─────────────────
# YAHAN CHANGE KIYA GAYA HAI:
if not os.path.exists('streamlit_app.py') or not os.path.exists('crew_agents.py'):
    from google.colab import files
    print('📂 Please upload streamlit_app.py AND crew_agents.py')
    files.upload()

# ── Kill stale remote endpoints via ngrok REST API ────────────────────────────
# pyngrok disconnect() only works when the local ngrok process is running.
# If the process is dead (e.g. runtime restart), the remote endpoint stays
# alive on ngrok's servers. We must DELETE it through the REST API directly.
def _kill_remote_ngrok_endpoints(token: str):
    api = 'https://api.ngrok.com'
    headers = {
        'Authorization': f'Bearer {token}',
        'Ngrok-Version': '2',
    }
    try:
        req = urllib.request.Request(f'{api}/endpoints', headers=headers)
        with urllib.request.urlopen(req, timeout=10) as r:
            endpoints = json.loads(r.read())
        for ep in endpoints.get('endpoints', []):
            eid = ep['id']
            del_req = urllib.request.Request(
                f'{api}/endpoints/{eid}', headers=headers, method='DELETE')
            try:
                urllib.request.urlopen(del_req, timeout=10)
                print(f'  🗑️  Deleted remote endpoint {ep.get("public_url", eid)}')
            except Exception as e:
                print(f'  ⚠️  Could not delete {eid}: {e}')
    except Exception as e:
        print(f'  ⚠️  REST API call failed: {e}')

print('🔄 Clearing any stale remote ngrok endpoints...')
_kill_remote_ngrok_endpoints(NGROK_TOKEN)
ngrok.kill()   # also stop local process if somehow still running
time.sleep(2)

# ── Configure ngrok and start Streamlit ──────────────────────────────────────
conf.get_default().auth_token = NGROK_TOKEN

threading.Thread(
    target=lambda: os.system(
        'streamlit run streamlit_app.py '
        '--server.port 8501 '
        '--server.headless true '
        '--server.enableCORS false '
        '--server.enableXsrfProtection false '
        '> /tmp/streamlit.log 2>&1'
    ), daemon=True
).start()
time.sleep(6)

tunnel = ngrok.connect(8501)
public_url = tunnel.public_url
print(f'🌐 Streamlit UI: {public_url}')
print('  Sidebar → paste Groq API key → Connect')
print('  Upload CSV files → click ▶️  Run Pipeline')

🔄 Clearing any stale remote ngrok endpoints...
  ⚠️  REST API call failed: HTTP Error 400: Bad Request
🌐 Streamlit UI: https://mutt-reconcile-lumpiness.ngrok-free.dev
  Sidebar → paste Groq API key → Connect
  Upload CSV files → click ▶️  Run Pipeline
